# Phase 3 — Lot Boundary Recovery & Land Use Filtering
**Ontario Distribution-Connected Solar Siting | 10+ MW Projects**

Recover original lot boundaries for Phase 2 developable lots, dump multi-part
geometries to single parts, clip against the county official plan land-use layer,
apply building setback exclusions (conditional: 150 m near REA zones, 30 m default),
filter to retain only rural/industrial designations, and intersect with parcels to
produce the final developable area parcels table.

**Input**
- `analysis.lots_developable_{county_slug}` — Phase 2 Step 2 output (lots after REA exclusion erase)
- `analysis.rea_exclusions_{county_slug}` — REA exclusion zones (Phase 2 Step 1)
- `analysis.aoi_county_{county_slug}` — county AOI (Phase 1 Step 1)

**Steps**
1. Recover original lot boundaries & extract parcels
2. Dump + land-use clip + designation filter (inspection only)
3. Building exclusion & final developable area parcels
4. Verification & interactive map

---
Change only `COUNTY_NAME` (and optionally `LANDUSE_COL`) in the Configuration cell to reproduce for any county.

## 0 · Configuration

In [37]:
# ─────────────────────────────────────────────────────────────────────────────
# PROJECT PARAMETERS — edit this cell only to run for a different county
# ─────────────────────────────────────────────────────────────────────────────

import os

COUNTY_NAME    = "Oxford"          # Must match adm_district_township.name_2
AOI_BUFFER_M   = 2000             # AOI buffer in metres (EPSG:5321)
MIN_POLYGON_HA = 30               # OEM polygon noise filter (hectares)
MIN_NET_ACRES  = 40               # Minimum net acres after REA exclusions
MIN_LOT_ACRES    = 40             # Minimum single-part lot fragment in acres
MIN_PARCEL_ACRES = 40             # Minimum parcel size in acres
LANDUSE_COL      = "descriptionlanduse"    # Land-use designation column (adjust per county)
BUILDING_BUFFER_REA     = 150     # Buffer (m) for buildings inside REA exclusion zones
BUILDING_BUFFER_DEFAULT = 30      # Buffer (m) for buildings outside REA exclusion zones

# Land-use designations suitable for solar siting (matched case-insensitively via ILIKE)
ALLOWED_LANDUSE = [
    "%rural%",
    "%rural industrial%",
    "%industrial%",
    "%employment%",
    "%quarry%",
]

EXCLUDED_LANDUSE = [
    "%agricultural%",
    "%residential%",
    "%urban%",
    "%village%",
    "%institutional%",
    "%commercial%",
    "%environmental%",
    "%conservation%",
    "%open space%",
    "%natural%",
    "%heritage%",
]

# PostGIS connection
PG_CONN = os.environ["POSTGRES_CONNECTION_STRING"]

# Output schema — tables are named per step and county automatically
OUTPUT_SCHEMA = "analysis"

# CRS
CRS_WGS84   = 4326
CRS_NAD83   = 4269   # NAD83 geographic — bridge CRS
CRS_ONTARIO = 5321   # NAD83(CSRS) / Ontario MNR Lambert — all calculations

## 1 · Environment Setup & Utilities

In [38]:
import warnings
warnings.filterwarnings("ignore")

import geopandas as gpd
import pandas as pd
import folium
from folium.plugins import MiniMap
from branca.colormap import linear
from sqlalchemy import create_engine, text

engine = create_engine(PG_CONN)

def read_postgis(sql: str, geom_col: str = "geom") -> gpd.GeoDataFrame:
    """Execute a PostGIS query and return a WGS84 GeoDataFrame."""
    gdf = gpd.read_postgis(sql, engine, geom_col=geom_col)
    if gdf.crs is None:
        gdf = gdf.set_crs(epsg=CRS_WGS84)
    return gdf.to_crs(epsg=CRS_WGS84)


def save_to_postgis(gdf: gpd.GeoDataFrame, table: str, label: str) -> None:
    """Write a GeoDataFrame to PostGIS in EPSG:5321 and create a GiST spatial index."""
    geom_col = gdf.geometry.name
    gdf.to_crs(epsg=CRS_ONTARIO).to_postgis(
        name=table,
        con=engine,
        schema=OUTPUT_SCHEMA,
        if_exists="replace",
        index=False,
        chunksize=500
    )
    with engine.connect() as conn:
        conn.execute(text(
            f"CREATE INDEX IF NOT EXISTS {table}_geom_idx "
            f"ON {OUTPUT_SCHEMA}.{table} USING GIST({geom_col})"
        ))
        conn.commit()
    print(f"  {label:35s} → {OUTPUT_SCHEMA}.{table} "
          f"({len(gdf):,} rows, EPSG:{CRS_ONTARIO}, GiST index)")


county_slug = COUNTY_NAME.lower().replace(" ", "_")

with engine.connect() as conn:
    print("PostGIS:", conn.execute(text("SELECT PostGIS_Full_Version()")).scalar())
print(f"County : {COUNTY_NAME} (slug: {county_slug})")

PostGIS: POSTGIS="3.6.1 0" [EXTENSION] PGSQL="170" GEOS="3.14.1-CAPI-1.20.5" SFCGAL="SFCGAL 2.2.0, CGAL 6.1, BOOST 1.79.0" PROJ="9.7.1 NETWORK_ENABLED=OFF URL_ENDPOINT=https://cdn.proj.org USER_WRITABLE_DIRECTORY=/tmp/proj DATABASE_PATH=/usr/local/pgsql/share/proj/proj.db" (compiled against PROJ 9.7.1) GDAL="GDAL 3.12.1 "Chicoutimi", released 2025/12/12" LIBXML="2.11.5" LIBJSON="0.17" LIBPROTOBUF="1.5.0" WAGYU="0.5.0 (Internal)" TOPOLOGY RASTER
County : Oxford (slug: oxford)


---
## Step 1A — Recover Original Lot Boundaries

Phase 2 stores lots with **eroded** geometry (after `ST_Difference` with REA
exclusions). For parcel extraction we need the **original** lot boundary from
`cadastre.ontario_lots`.

**Strategy**: Use `ST_PointOnSurface` on each developable lot's geometry to
create a guaranteed-interior representative point, then find the original lot
in `cadastre.ontario_lots` that contains that point. This avoids false matches
from edge/boundary overlaps that `ST_Intersects` on full polygons would produce.

**Why `ST_PointOnSurface`**: Unlike `ST_Centroid`, which can fall outside a
concave polygon, `ST_PointOnSurface` is guaranteed to lie within the geometry.

In [39]:
county_slug = COUNTY_NAME.lower().replace(" ", "_")

gdf_lots_boundary = read_postgis(f"""
SELECT
    so.fid,
    so.lot_ident,
    so.concession_ident,
    so.geographic_township_name,
    so.objectid,
    so.refreshed_datetime,
    so.name,
    ST_Area(ST_Transform(so.geom, {CRS_ONTARIO})) / 4046.856 AS lot_acre,
    ST_Transform(so.geom, {CRS_WGS84}) AS geom
FROM cadastre.ontario_lots AS so
INNER JOIN {OUTPUT_SCHEMA}.lots_developable_{county_slug} AS dl
    ON ST_Intersects(
        so.geom,
        ST_Transform(ST_PointOnSurface(dl.geom), ST_SRID(so.geom))
    )
""")

print(f"Step 1A — Original lot boundaries for '{COUNTY_NAME}':")
print(f"  Lot count    : {len(gdf_lots_boundary):,}")
print(f"  Total acreage: {gdf_lots_boundary['lot_acre'].sum():,.1f} ac")

save_to_postgis(gdf_lots_boundary, f"selected_lots_boundary_{county_slug}",
                "Step 1A — Original lot boundaries")

Step 1A — Original lot boundaries for 'Oxford':
  Lot count    : 1,104
  Total acreage: 216,969.6 ac
  Step 1A — Original lot boundaries   → analysis.selected_lots_boundary_oxford (1,104 rows, EPSG:5321, GiST index)


---
## Step 1B — Extract Parcels Within Lot Boundaries

Extract parcels from `cadastre.parcels_{county_slug}` that fall within the
dissolved lot boundary envelope. The `&&` operator drives the GiST index
for a fast bounding-box pre-filter, then `ST_Intersects` confirms true
geometric overlap. Filter to parcels ≥ `MIN_PARCEL_ACRES` (40 acres).

In [40]:
county_slug = COUNTY_NAME.lower().replace(" ", "_")

gdf_parcels = read_postgis(f"""
WITH boundary AS (
    SELECT ST_Transform(
        ST_Union(lb.geom),
        (SELECT ST_SRID(geom) FROM cadastre.parcels_{county_slug} LIMIT 1)
    ) AS geom
    FROM {OUTPUT_SCHEMA}.selected_lots_boundary_{county_slug} AS lb
)
SELECT
    -- p.parcel_id,
    p.ogf_id,
    p.assessment_roll_number,
    ST_Area(ST_Transform(p.geom, {CRS_ONTARIO})) / 4046.856 AS parcel_acre,
    ST_Transform(p.geom, {CRS_WGS84}) AS geom
FROM cadastre.parcels_{county_slug} AS p
JOIN boundary b
    ON p.geom && b.geom
   AND ST_Intersects(p.geom, b.geom)
WHERE ST_Area(ST_Transform(p.geom, {CRS_ONTARIO})) / 4046.856 > {MIN_PARCEL_ACRES}
""")

print(f"Step 1B — Parcels within lot boundaries for '{COUNTY_NAME}':")
print(f"  Parcel count  : {len(gdf_parcels):,}")
print(f"  Total acreage : {gdf_parcels['parcel_acre'].sum():,.1f} ac")

save_to_postgis(gdf_parcels, f"selected_parcels_{county_slug}",
                "Step 1B — Parcels within lot boundaries")

Step 1B — Parcels within lot boundaries for 'Oxford':
  Parcel count  : 2,243
  Total acreage : 212,471.5 ac
  Step 1B — Parcels within lot boundaries → analysis.selected_parcels_oxford (2,243 rows, EPSG:5321, GiST index)


---
## Step 2 — Dump, Land-Use Clip & Designation Filter (Inspection Only)

Combines three operations in a single CTE chain:
1. **ST_Dump** each `lots_developable` geometry into single parts, filter ≥ `MIN_LOT_ACRES`
2. **Clip** dumped fragments against `official_plans.land_use_{county_slug}` to attach the
   land-use designation
3. **Filter** by allowed/excluded land-use designations and re-compute acreage

> **No DB save** — this GeoDataFrame is for inspection only. The same CTE
> logic is embedded inline in Step 3B's query to produce the final output.

In [41]:
county_slug = COUNTY_NAME.lower().replace(" ", "_")

# Build ILIKE filter — double %% to escape psycopg2 parameter markers
allowed_clause = " OR ".join(
    f"land_use_designation ILIKE '{p}'" for p in [x.replace("%", "%%") for x in ALLOWED_LANDUSE]
)
excluded_clause = " AND ".join(
    f"land_use_designation NOT ILIKE '{p}'" for p in [x.replace("%", "%%") for x in EXCLUDED_LANDUSE]
)

gdf_lots_filtered = read_postgis(f"""
WITH dumped AS (
    SELECT
        l.lot_ident,
        l.concession_ident,
        l.geographic_township_name,
        l.acre_total,
        l.grid_capacity_mw,
        l.voltage_3ph,
        l.station_name,
        l.feeder_name,
        l.ldc_name,
        l.acre_net   AS acre_net_phase2,
        l.pct_remaining,
        (ST_Dump(l.geom)).geom AS geom
    FROM {OUTPUT_SCHEMA}.lots_developable_{county_slug} l
),
dumped_filtered AS (
    SELECT *
    FROM dumped
    WHERE ST_Area(geom) / 4046.856 > {MIN_LOT_ACRES}
),
landuse_clipped AS (
    SELECT
        d.lot_ident,
        d.concession_ident,
        d.geographic_township_name,
        d.acre_total,
        d.grid_capacity_mw,
        d.voltage_3ph,
        d.station_name,
        d.feeder_name,
        d.ldc_name,
        d.acre_net_phase2,
        d.pct_remaining,
        lu.{LANDUSE_COL} AS land_use_designation,
        ST_MakeValid(ST_Intersection(d.geom, ST_Transform(lu.geom, {CRS_ONTARIO}))) AS geom
    FROM dumped_filtered d
    JOIN official_plans.land_use_{county_slug} AS lu
        ON ST_Intersects(d.geom, ST_Transform(lu.geom, {CRS_ONTARIO}))
       AND ST_Area(ST_Intersection(d.geom, ST_Transform(lu.geom, {CRS_ONTARIO}))) > 0
)
SELECT
    lot_ident,
    concession_ident,
    geographic_township_name,
    acre_total,
    grid_capacity_mw,
    voltage_3ph,
    station_name,
    feeder_name,
    ldc_name,
    acre_net_phase2,
    pct_remaining,
    land_use_designation,
    ROUND((ST_Area(geom) / 4046.856)::numeric, 2) AS acre_landuse,
    ST_Transform(geom, {CRS_WGS84}) AS geom
FROM landuse_clipped
WHERE ST_Area(geom) / 4046.856 > {MIN_LOT_ACRES}
  AND ({allowed_clause})
  AND ({excluded_clause})
ORDER BY acre_landuse DESC
""")

print(f"Step 2 — Dump + land-use clip + designation filter (inspection):")
print(f"  Lots after filter     : {len(gdf_lots_filtered):,}")
print(f"  Total filtered acreage: {gdf_lots_filtered['acre_landuse'].sum():,.1f} ac")
print(f"  Designations found    : {gdf_lots_filtered['land_use_designation'].unique().tolist()}")

Step 2 — Dump + land-use clip + designation filter (inspection):
  Lots after filter     : 38
  Total filtered acreage: 3,315.7 ac
  Designations found    : ['Quarry Area', 'Traditional Industrial', 'Industrial', 'Prime Industrial', 'Rural Buffer']


---
## Step 3A — Conditional Building Buffer & Dissolve

Build the building setback exclusion layer with **conditional buffers**:
- Buildings inside `analysis.rea_exclusions_{county_slug}` → **150 m** (near environmentally sensitive areas)
- Buildings outside REA zones → **30 m** (standard setback)

The REA CTE produces a single dissolved geometry, so `CROSS JOIN rea` adds
exactly one row per building (no cartesian explosion). Dissolved into a
single exclusion geometry via `ST_Union` + `ST_MakeValid`.

In [42]:
county_slug = COUNTY_NAME.lower().replace(" ", "_")

gdf_building_exclusion = read_postgis(f"""
WITH rea AS (
    SELECT ST_Union(geom) AS geom
    FROM {OUTPUT_SCHEMA}.rea_exclusions_{county_slug}
),
buildings_proj AS (
    SELECT
        ST_Transform(b.geom, {CRS_ONTARIO}) AS geom
    FROM cadastre.buildings_{county_slug} b
),
buildings_buffered AS (
    SELECT
        CASE
            WHEN ST_Intersects(bp.geom, r.geom)
            THEN ST_Buffer(bp.geom, {BUILDING_BUFFER_REA})
            ELSE ST_Buffer(bp.geom, {BUILDING_BUFFER_DEFAULT})
        END AS geom
    FROM buildings_proj bp
    CROSS JOIN rea r
),
dissolved AS (
    SELECT ST_MakeValid(ST_Union(geom)) AS geom
    FROM buildings_buffered
)
SELECT ST_Transform(geom, {CRS_WGS84}) AS geom
FROM dissolved
""")

print(f"Step 3A — Building exclusion zone for '{COUNTY_NAME}':")
print(f"  Geometry type : {gdf_building_exclusion.geometry.geom_type.iloc[0]}")
print(f"  Total area    : {gdf_building_exclusion.to_crs(epsg=CRS_ONTARIO).geometry.area.sum() / 10000:,.1f} ha")

save_to_postgis(gdf_building_exclusion, f"building_exclusion_{county_slug}",
                "Step 3A — Building buffer exclusion zone")

Step 3A — Building exclusion zone for 'Oxford':
  Geometry type : MultiPolygon
  Total area    : 44,057.3 ha
  Step 3A — Building buffer exclusion zone → analysis.building_exclusion_oxford (1 rows, EPSG:5321, GiST index)


---
## Step 3B — Final Developable Area Parcels

Single CTE chain producing the **final Phase 3 output table**:
1. **Dump** `lots_developable` to single parts → filter ≥ `MIN_LOT_ACRES`
2. **Clip** against the land-use layer
3. **Filter** by allowed/excluded land-use designations
4. **Subtract** building exclusion layer via `ST_Difference`
5. **Intersect** with `selected_parcels` → filter ≥ `MIN_PARCEL_ACRES`

`LEFT JOIN` + `COALESCE` in `lots_no_buildings` preserves lot fragments
with no building overlap. Parcel intersection reads from
`analysis.selected_parcels_{slug}` (already in EPSG:5321).

In [43]:
county_slug = COUNTY_NAME.lower().replace(" ", "_")

# Build ILIKE filter — double %% to escape psycopg2 parameter markers
allowed_clause = " OR ".join(
    f"land_use_designation ILIKE '{p}'" for p in [x.replace("%", "%%") for x in ALLOWED_LANDUSE]
)
excluded_clause = " AND ".join(
    f"land_use_designation NOT ILIKE '{p}'" for p in [x.replace("%", "%%") for x in EXCLUDED_LANDUSE]
)

gdf_final = read_postgis(f"""
WITH dumped AS (
    SELECT
        l.lot_ident,
        l.concession_ident,
        l.geographic_township_name,
        l.grid_capacity_mw,
        l.voltage_3ph,
        l.station_name,
        l.feeder_name,
        l.ldc_name,
        (ST_Dump(l.geom)).geom AS geom
    FROM {OUTPUT_SCHEMA}.lots_developable_{county_slug} l
),
dumped_filtered AS (
    SELECT *
    FROM dumped
    WHERE ST_Area(geom) / 4046.856 > {MIN_LOT_ACRES}
),
landuse_clipped AS (
    SELECT
        d.lot_ident,
        d.concession_ident,
        d.geographic_township_name,
        d.grid_capacity_mw,
        d.voltage_3ph,
        d.station_name,
        d.feeder_name,
        d.ldc_name,
        lu.{LANDUSE_COL}          AS land_use_designation,
        -- lu.title_en               AS op_title,
        -- lu.sched_b9_rural,
        ST_MakeValid(ST_Intersection(d.geom, ST_Transform(lu.geom, {CRS_ONTARIO}))) AS geom
    FROM dumped_filtered d
    JOIN official_plans.land_use_{county_slug} AS lu
        ON ST_Intersects(d.geom, ST_Transform(lu.geom, {CRS_ONTARIO}))
       AND ST_Area(ST_Intersection(d.geom, ST_Transform(lu.geom, {CRS_ONTARIO}))) > 0
),
landuse_filtered AS (
    SELECT *
    FROM landuse_clipped
    WHERE ST_Area(geom) / 4046.856 > {MIN_LOT_ACRES}
      AND ({allowed_clause})
      AND ({excluded_clause})
),
bldg_excl AS (
    SELECT ST_Union(geom) AS geom
    FROM {OUTPUT_SCHEMA}.building_exclusion_{county_slug}
),
lots_no_buildings AS (
    SELECT
        lf.lot_ident,
        lf.concession_ident,
        lf.geographic_township_name,
        lf.grid_capacity_mw,
        lf.voltage_3ph,
        lf.station_name,
        lf.feeder_name,
        lf.ldc_name,
        lf.land_use_designation,
        -- lf.op_title,
        -- lf.sched_b9_rural,
        ST_MakeValid(ST_Difference(
            lf.geom,
            COALESCE(be.geom, ST_GeomFromText('GEOMETRYCOLLECTION EMPTY', {CRS_ONTARIO}))
        )) AS geom
    FROM landuse_filtered lf
    LEFT JOIN bldg_excl be ON ST_Intersects(lf.geom, be.geom)
    WHERE ST_Area(ST_Difference(
            lf.geom,
            COALESCE(be.geom, ST_GeomFromText('GEOMETRYCOLLECTION EMPTY', {CRS_ONTARIO}))
          )) / 4046.856 > 0
)
SELECT
    l.lot_ident,
    l.concession_ident,
    l.geographic_township_name,
    l.grid_capacity_mw,
    l.voltage_3ph,
    l.station_name,
    l.feeder_name,
    l.ldc_name,
    l.land_use_designation,
    -- l.op_title,
    -- l.sched_b9_rural,
    -- p.parcel_id         AS parcel_id,
    p.ogf_id     AS parcel_ogf_id,
    p.assessment_roll_number,
    ROUND(p.parcel_acre::numeric, 2)  AS parcel_acre,
    ROUND((ST_Area(
        ST_Intersection(l.geom, p.geom)
    ) / 4046.856)::numeric, 2) AS net_developable_acre,
    ST_Transform(
        ST_MakeValid(ST_Intersection(l.geom, p.geom)),
        {CRS_WGS84}
    ) AS geom
FROM lots_no_buildings l
JOIN {OUTPUT_SCHEMA}.selected_parcels_{county_slug} p
    ON ST_Intersects(l.geom, p.geom)
   AND ST_Area(ST_Intersection(l.geom, p.geom)) > 0
WHERE ST_Area(
        ST_Intersection(l.geom, p.geom)
      ) / 4046.856 > {MIN_PARCEL_ACRES}
ORDER BY net_developable_acre DESC
""")

print(f"Step 3B — Final developable area parcels for '{COUNTY_NAME}':")
print(f"  Parcel count         : {len(gdf_final):,}")
print(f"  Total gross ac       : {gdf_final['parcel_acre'].sum():,.1f} ac")
print(f"  Total net dev ac     : {gdf_final['net_developable_acre'].sum():,.1f} ac")
print(f"  Columns              : {list(gdf_final.columns)}")

save_to_postgis(gdf_final, f"developable_area_parcels_{county_slug}",
                "Step 3B — Final developable area parcels")

Step 3B — Final developable area parcels for 'Oxford':
  Parcel count         : 24
  Total gross ac       : 5,168.9 ac
  Total net dev ac     : 1,678.9 ac
  Columns              : ['lot_ident', 'concession_ident', 'geographic_township_name', 'grid_capacity_mw', 'voltage_3ph', 'station_name', 'feeder_name', 'ldc_name', 'land_use_designation', 'parcel_ogf_id', 'assessment_roll_number', 'parcel_acre', 'net_developable_acre', 'geom']
  Step 3B — Final developable area parcels → analysis.developable_area_parcels_oxford (24 rows, EPSG:5321, GiST index)


---
## Step 3C — Developable Parcel Boundaries

Extract the **original parcel boundaries** from Step 1B (`selected_parcels_{slug}`)
that correspond to the final developable area parcels from Step 3B. Uses a simple
attribute join on `parcel_id` — no spatial join required.

Saved as `developable_parcels_{slug}` for clean parcel outlines on the map.

In [44]:
county_slug = COUNTY_NAME.lower().replace(" ", "_")

# Get distinct parcel IDs from the final developable area parcels
dev_parcel_ids = gdf_final["parcel_ogf_id"].dropna().unique().tolist()
print(f"Distinct parcel IDs in developable_area_parcels: {len(dev_parcel_ids)}")

gdf_dev_parcels = read_postgis(f"""
SELECT
    -- sp.parcel_id        AS parcel_id,
    sp.ogf_id,
    sp.assessment_roll_number,
    sp.parcel_acre,
    ST_Transform(sp.geom, {CRS_WGS84}) AS geom
FROM {OUTPUT_SCHEMA}.selected_parcels_{county_slug} sp
INNER JOIN {OUTPUT_SCHEMA}.developable_area_parcels_{county_slug} dap
    ON sp.ogf_id = dap.parcel_ogf_id
GROUP BY sp.ogf_id, sp.assessment_roll_number, sp.parcel_acre, sp.geom
""")

print(f"Step 3C — Developable parcel boundaries for '{COUNTY_NAME}':")
print(f"  Parcel count  : {len(gdf_dev_parcels):,}")
print(f"  Total acreage : {gdf_dev_parcels['parcel_acre'].sum():,.1f} ac")

save_to_postgis(gdf_dev_parcels, f"developable_parcels_{county_slug}",
                "Step 3C — Developable parcel boundaries")

Distinct parcel IDs in developable_area_parcels: 21
Step 3C — Developable parcel boundaries for 'Oxford':
  Parcel count  : 21
  Total acreage : 3,832.8 ac
  Step 3C — Developable parcel boundaries → analysis.developable_parcels_oxford (21 rows, EPSG:5321, GiST index)


---
## Verification & Summary Statistics

Consolidated summary of all Phase 3 steps, showing the filtering funnel from
lot boundaries → parcels → land-use filter → building exclusion → final output.

In [45]:
county_slug = COUNTY_NAME.lower().replace(" ", "_")

print(f"Phase 3 — Summary for '{COUNTY_NAME}'")
print(f"{'─' * 55}")
print(f"  Step 1A — Lot boundaries          : {len(gdf_lots_boundary):,} lots, "
      f"{gdf_lots_boundary['lot_acre'].sum():,.1f} ac")
print(f"  Step 1B — Parcels in lot envelope  : {len(gdf_parcels):,} parcels, "
      f"{gdf_parcels['parcel_acre'].sum():,.1f} ac")
print(f"  Step 2  — After land-use filter    : {len(gdf_lots_filtered):,} fragments, "
      f"{gdf_lots_filtered['acre_landuse'].sum():,.1f} ac")
print(f"            Designations             : {gdf_lots_filtered['land_use_designation'].unique().tolist()}")
print(f"  Step 3A — Building exclusion area  : "
      f"{gdf_building_exclusion.to_crs(epsg=CRS_ONTARIO).geometry.area.sum() / 10000:,.1f} ha")
print(f"  Step 3B — Final developable areas  : {len(gdf_final):,} parcels, "
      f"{gdf_final['net_developable_acre'].sum():,.1f} ac (net)")
print(f"  Step 3C — Developable parcels      : {len(gdf_dev_parcels):,} parcels, "
      f"{gdf_dev_parcels['parcel_acre'].sum():,.1f} ac (gross)")
print()

# List Phase 3 analysis tables
print("Phase 3 tables in analysis schema:")
with engine.connect() as conn:
    rows = conn.execute(text(f"""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = '{OUTPUT_SCHEMA}'
          AND table_name LIKE '%%{county_slug}%%'
        ORDER BY table_name
    """)).fetchall()
    for r in rows:
        print(f"  {OUTPUT_SCHEMA}.{r[0]}")

Phase 3 — Summary for 'Oxford'
───────────────────────────────────────────────────────
  Step 1A — Lot boundaries          : 1,104 lots, 216,969.6 ac
  Step 1B — Parcels in lot envelope  : 2,243 parcels, 212,471.5 ac
  Step 2  — After land-use filter    : 38 fragments, 3,315.7 ac
            Designations             : ['Quarry Area', 'Traditional Industrial', 'Industrial', 'Prime Industrial', 'Rural Buffer']
  Step 3A — Building exclusion area  : 44,057.3 ha
  Step 3B — Final developable areas  : 24 parcels, 1,678.9 ac (net)
  Step 3C — Developable parcels      : 21 parcels, 3,832.8 ac (gross)

Phase 3 tables in analysis schema:
  analysis.aoi_county_oxford
  analysis.building_exclusion_oxford
  analysis.developable_area_parcels_oxford
  analysis.developable_parcels_oxford
  analysis.lots_developable_oxford
  analysis.oem_county_oxford
  analysis.rea_exclusions_oxford
  analysis.selected_lots_boundary_oxford
  analysis.selected_lots_oxford
  analysis.selected_parcels_oxford


---
## Interactive Folium Map

Layers:
- **Developable parcels** (blue outline) — original parcel boundaries from Step 3C
- **Final developable areas** (green gradient) — colored by `net_developable_acre`
- **Building exclusion zones** (semi-transparent dark)
- **REA exclusion zones** (semi-transparent green/red, context)

In [46]:
# ── Interactive map ─────────────────────────────────────────────────────────
county_slug = COUNTY_NAME.lower().replace(" ", "_")

# Load context layers
gdf_rea = read_postgis(f"""
    SELECT geom FROM {OUTPUT_SCHEMA}.rea_exclusions_{county_slug}
""")

# Map center from final parcels
center = [
    gdf_final.geometry.centroid.y.mean(),
    gdf_final.geometry.centroid.x.mean()
]

# Colormap — net_developable_acre: light green (small) → dark green (large)
acre_vals = gdf_final["net_developable_acre"].dropna()
acre_cmap = linear.YlGn_09.scale(acre_vals.min(), acre_vals.max())
acre_cmap.caption = "Net Developable Area (acres)"

m = folium.Map(location=center, zoom_start=10, tiles=None)
MiniMap(toggle_display=True).add_to(m)

# Esri World Imagery basemap (reliable XYZ tiles with Ontario coverage)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, Maxar, Earthstar Geographics, and the GIS User Community",
    name="Esri World Imagery",
    max_zoom=19,
).add_to(m)

# Layer 1 — Developable parcel boundaries (reference outline from Step 3C)
folium.GeoJson(
    gdf_dev_parcels.__geo_interface__,
    name="Developable parcels (boundary)",
    style_function=lambda _: {
        "fillColor": "transparent", "color": "#d83d00",
        "weight": 2.5, "fillOpacity": 0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["ogf_id", "parcel_acre"],
        aliases=["Parcel ID", "Parcel Area (ac)"],
        localize=True
    )
).add_to(m)

# Layer 2 — REA exclusion zones (context)
folium.GeoJson(
    gdf_rea.__geo_interface__,
    name="REA exclusion zones",
    style_function=lambda _: {
        "fillColor": "#89dd59ff", "color": "#0e4908",
        "weight": 0.5, "fillOpacity": 0.2
    }
).add_to(m)

# Layer 3 — Building exclusion zones
folium.GeoJson(
    gdf_building_exclusion.__geo_interface__,
    name="Building exclusion zones",
    style_function=lambda _: {
        "fillColor": "#5e473b", "color": "#382619",
        "weight": 0.5, "fillOpacity": 0.2
    }
).add_to(m)

# Layer 4 — Final developable areas (colored by net_developable_acre)
folium.GeoJson(
    gdf_final.__geo_interface__,
    name="Developable areas (Phase 3)",
    style_function=lambda feat: {
        "fillColor": acre_cmap(feat["properties"].get("net_developable_acre") or 0),
        "color": "#5e221a", "weight": 0.8, "fillOpacity": 0.75
    },
    highlight_function=lambda _: {
        "color": "#0d3b0d", "weight": 2.5, "fillOpacity": 0.9
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["lot_ident", "concession_ident", "geographic_township_name",
                "grid_capacity_mw", "voltage_3ph", "station_name", "feeder_name", "ldc_name",
                "land_use_designation",
                "parcel_ogf_id",
                "parcel_acre", "net_developable_acre"],
        aliases=["Lot", "Concession", "Township",
                 "Grid Capacity (MW)", "3-Phase Voltage", "Station", "Feeder", "LDC",
                 "Land Use",
                 "Parcel ID",
                 "Parcel Area (ac)", "Net Developable Area (ac)"],
        localize=True, sticky=True
    )
).add_to(m)

acre_cmap.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

out_html = f"output/map_phase3_{county_slug}.html"
m.save(out_html)
print(f"Saved: {out_html}")

Saved: output/map_phase3_oxford.html
